# Week 12 Lecture: The Professional ML Workflow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bradleyboehmke/uc-bana-4080/blob/main/notebooks/tuesday-your-turn/week-12-lecture.ipynb)

This week you'll stop peeking at the test set. You'll use cross-validation to score models honestly, `GridSearchCV` to tune hyperparameters, and a single pipeline to impute, encode, and scale features without leaking information.

## How to use this notebook

This notebook accompanies the Week 12 Tuesday lecture. You can:

- **Follow along during class**: run each cell as the instructor discusses it
- **Work the Your Turn exercises** when prompted
- **Review after class**: rerun and experiment with the code

## Setup

Everything we need is already installed in Colab. Today's data is the cleaned **Ames housing** data, loaded straight from the course GitHub repository. Each row is a home sold in Ames, Iowa, and `SalePrice` is what we'll predict.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.neighbors import KNeighborsRegressor

In [ ]:
url = "https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/data/ames_clean.csv"
ames = pd.read_csv(url)

print(ames.shape)
ames.head()

---

## 1. Cross-Validation

### The peeking problem

Last week we chose settings like `max_depth` by checking the test set each time. In class, the Optimism Gap chart showed what that habit does: the best test score keeps climbing, while the real-world score of that same model climbs far less.

Here's the same habit on Ames. We split the data three ways:

- **Training set** (60%): what the trees learn from
- **Test set** (100 homes): the set we keep checking to pick a depth
- **New homes** (the rest): homes that play no part in the choice, standing in for the future

In [ ]:
features = ['GrLivArea', 'LotArea', 'YearRemodAdd', 'TotRmsAbvGrd', 'GarageArea', 'Fireplaces']
X, y = ames[features], ames['SalePrice']

X_train, X_rest, y_train, y_rest = train_test_split(X, y, train_size=0.6, random_state=0)
X_test, X_new, y_test, y_new = train_test_split(X_rest, y_rest, train_size=100, random_state=0)

print(f"Train: {len(X_train)} homes | Test: {len(X_test)} homes | New: {len(X_new)} homes")

Try 20 depths, score each one on the same 100 test homes, and keep the winner. Then score that winner on the new homes.

In [ ]:
test_rmse = {}   # try 20 depths, scoring each on the same 100 test homes
for depth in range(1, 21):
    tree = DecisionTreeRegressor(max_depth=depth, random_state=42).fit(X_train, y_train)
    test_rmse[depth] = root_mean_squared_error(y_test, tree.predict(X_test))
best_depth = min(test_rmse, key=test_rmse.get)

# score the chosen depth on homes that played no part in the choice
best_tree = DecisionTreeRegressor(max_depth=best_depth, random_state=42).fit(X_train, y_train)
new_rmse = root_mean_squared_error(y_new, best_tree.predict(X_new))

print(f"Depth chosen by the test set:   {best_depth}")
print(f"What you report (test RMSE):    ${test_rmse[best_depth]:,.0f}")
print(f"What you get (new-home RMSE):   ${new_rmse:,.0f}")

The RMSE you would report is lower (better) than the RMSE on homes you never looked at. That's the optimism gap, on real data.

### What the test set hid

Two numbers don't show *why*. Let's score all 20 depths on both the test set and the new homes. In a real project you would only ever see the orange line.

In [ ]:
depths = range(1, 21)
new_home_rmse = [
    root_mean_squared_error(
        y_new,
        DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_train, y_train).predict(X_new))
    for d in depths
]

plt.figure(figsize=(9, 4))
plt.plot(depths, [test_rmse[d] for d in depths], marker="o", color="#e58139", label="Test set (what you see)")
plt.plot(depths, new_home_rmse, marker="o", color="#399de5", label="New homes (hidden in real life)")
plt.axvline(best_depth, color="gray", linestyle=":", label=f"Depth chosen by the test set ({best_depth})")
plt.xlabel("Tree max_depth")
plt.ylabel("RMSE ($, lower is better)")
plt.xticks(list(depths))
plt.legend()
plt.show()

truly_best = min(depths, key=lambda d: new_home_rmse[d - 1])
print(f"Best depth on the new homes: {truly_best}")

Depth 9 only looked best because its errors happened to be small on those 100 test homes. Peeking inflates the score **and** can steer you to the wrong model.

> **🔒 The takeaway:** the test set can't be used to make modeling decisions. Lock it away for one job: the **final generalization assessment**, run once, after every choice is made. That leaves a gap: we still need to compare models, pick hyperparameters, and estimate performance using **only the training data**. Cross-validation fills that gap.

### K-fold cross-validation

**K-fold cross-validation** splits the **training data** into K folds (here, 5). Each round, one fold sits out as the validation set, the model trains on the other four, and it's scored on the fold that sat out. The **CV score** is the average of the K rounds. The test set never takes part.

Same Ames features, now with a depth-6 tree. Split first, lock the test set away, then cross-validate on the training set only.

In [ ]:
# Step 1: split first, then lock the test set away
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 2: 5-fold CV on the TRAINING set only
tree = DecisionTreeRegressor(max_depth=6, random_state=42)
cv_scores = cross_val_score(tree, X_train, y_train, cv=5,
                            scoring='neg_root_mean_squared_error')
cv_rmse = -cv_scores   # scikit-learn reports negative RMSE so that higher is always better

print("RMSE by fold:", ", ".join(f"${v:,.0f}" for v in cv_rmse))
print(f"CV RMSE: ${cv_rmse.mean():,.0f} ± ${cv_rmse.std():,.0f}")

**Reading the output:**

- **The minus sign.** scikit-learn's scorers treat *higher as better*, so error metrics come back negative (`neg_root_mean_squared_error`). Flip the sign to get ordinary RMSE.
- **The mean** is our estimate of how a depth-6 tree will do on new homes.
- **The std** shows how much the score moves depending on which homes are held out. A big std means one validation score on its own could be misleading.

### 🎯 Your Turn: How Lucky Was Your Split?

In class we saw that a single train/test split's score depends a lot on *which* homes land in the test set. Try it yourself:

1. For each `random_state` from **0 to 9**, split `X` and `y` 80/20, fit a `DecisionTreeRegressor(max_depth=6, random_state=42)` on the training part, and print its **test RMSE**.
2. What are the lowest and highest test RMSEs you got? How does that range compare with the CV mean ± std above?
3. If you could report only one number to your manager, would you trust one split's test RMSE or the CV mean? Why?

*Hint: put the `train_test_split` call inside a `for seed in range(10):` loop and use `random_state=seed`. `root_mean_squared_error(y_true, y_pred)` gives the RMSE.*

In [ ]:
# Your code here

*Next up in class: using cross-validation to tune hyperparameters.*

---

## 2. Hyperparameter Tuning with GridSearchCV

*Coming soon: this section is added with the lecture's hyperparameter tuning slides.*

---

## 3. Feature Engineering & Pipelines

*Coming soon: this section is added with the lecture's feature engineering slides.*

---

## 🧾 What You Learned

| Code | What it does |
|------|--------------|
| `cross_val_score(model, X, y, cv=5, scoring=...)` | Score a model on 5 train/validation splits of the training data |
| `GridSearchCV(model, param_grid, cv=5)` | Try every combination in a grid, scoring each with cross-validation |
| `ColumnTransformer([...])` | Apply different preprocessing to different columns |
| `Pipeline([...])` | Chain preprocessing and a model so each CV fold fits them on its own training data |